In [67]:
from dotenv import load_dotenv
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain.document_loaders import TextLoader

load_dotenv(override=True)

openai = ChatOpenAI(model_name='gpt-4o-mini') | StrOutputParser()

# SequentialChain

In [68]:
# Carregar os documentos
loader = TextLoader('base_conhecimento_locacao.txt')
documents = loader.load()
# Carregar histórico de conversas
historico_conversas = """Cliente: Preciso saber como está a locação do Mateus. Chatbot: Você já consultou
                         quanto ele gastou no cartão corporativo? Cliente: Ainda não, só sei que ele pegou um Prisma."""
# Pergunta do cliente
pergunta = "qual a situação do carro alugado pela juliana?"

In [69]:
inputs = {
    "context": "\n".join(doc.page_content for doc in documents),
    "question": pergunta,
    "historico": historico_conversas
}

In [70]:
prompt_base_conhecimento = PromptTemplate(
    input_variables=["context", "question"],
    template="""Use o seguinte contexto para responder à pergunta. 
    Responda apenas com base nas informações fornecidas.
    Não forneceça instruções de procedimento já realizados.
    Não utilize informações externas ao contexto:
    Contexto: {context}
    Pergunta: {question}"""
)
prompt_historico_conversas = PromptTemplate(
    input_variables=["historico", "question"],
    template="""Use o histórico de conversas para responder à pergunta. 
    Responda apenas com base nas informações fornecidas. 
    Não forneceça instruções de procedimento já realizados.
    Não utilize informações externas ao contexto:
    Histórico: {historico}
    Pergunta: {question}"""
)
prompt_final = PromptTemplate(
    input_variables=["resposta_base_conhecimento", "resposta_historico_conversas"],
    template="""Combine as seguintes respostas para gerar uma resposta final,
    mas não forneça instruções de procedimentos já realizados:
    Resposta da base de conhecimento: {resposta_base_conhecimento}
    Resposta do histórico de conversas: {resposta_historico_conversas}"""
)

In [71]:
# Definir as cadeias  
chain_base_conhecimento = prompt_base_conhecimento | openai
chain_historico_conversas = prompt_historico_conversas | openai
chain_final = prompt_final | openai

In [72]:
# Passando dados e executando
resultado_base_conhecimento = chain_base_conhecimento.invoke({"context": inputs["context"], "question": inputs["question"]})
resultado_historico_conversas = chain_historico_conversas.invoke({"historico": inputs["historico"], "question": inputs["question"]})


In [73]:
print(resultado_base_conhecimento)

A locação do carro alugado pela Juliana está ativa, com devolução prevista para 20/10/2026. Até o momento, os gastos no cartão corporativo totalizam R$ 615,00, e não há multas registradas. O saldo disponível no cartão é de R$ 385,00.


In [74]:
print(resultado_historico_conversas)

Não há informações disponíveis sobre a situação do carro alugado pela Juliana no histórico de conversas.


In [75]:
resultado_final = chain_final.invoke({"resposta_base_conhecimento": resultado_base_conhecimento, 
                                      "resposta_historico_conversas": resultado_historico_conversas})

In [76]:
print("Resultado Base de Conhecimento:\n", resultado_base_conhecimento)
print("----")
print("Resultado Histórico de Conversas:\n", resultado_historico_conversas)

Resultado Base de Conhecimento:
 A locação do carro alugado pela Juliana está ativa, com devolução prevista para 20/10/2026. Até o momento, os gastos no cartão corporativo totalizam R$ 615,00, e não há multas registradas. O saldo disponível no cartão é de R$ 385,00.
----
Resultado Histórico de Conversas:
 Não há informações disponíveis sobre a situação do carro alugado pela Juliana no histórico de conversas.


In [77]:
print(resultado_final)

A locação do carro alugado pela Juliana está ativa, com devolução prevista para 20/10/2026. Até o momento, os gastos no cartão corporativo totalizam R$ 615,00, não há multas registradas e o saldo disponível no cartão é de R$ 385,00. Não há informações adicionais sobre a situação do carro no histórico de conversas.
